<a href="https://colab.research.google.com/github/gouthamsankar-clg/AI-Lab/blob/main/Lab_6/Propositional_Logic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re

def parse(s):
    t = re.findall(r'<=>|=>|\w+|[~&|()]', s)
    pos = [0]
    peek = lambda: t[pos[0]] if pos[0] < len(t) else None
    def eat():
        pos[0] += 1
        return t[pos[0] - 1] if pos[0] - 1 < len(t) else None

    def iff():
        l = imp()
        while peek() == '<=>':
            eat(); l = ('iff', l, imp())
        return l
    def imp():
        l = or_()
        if peek() == '=>':
            eat(); return ('implies', l, imp())
        return l
    def or_():
        l = and_()
        while peek() == '|':
            eat(); l = ('or', l, and_())
        return l
    def and_():
        l = not_()
        while peek() == '&':
            eat(); l = ('and', l, not_())
        return l
    def not_():
        tok = eat()
        if tok == '~':
            return ('not', not_())
        if tok == '(':
            e = iff()
            if eat() != ')':
                raise ValueError("Missing )")
            return e
        if tok is None or not re.fullmatch(r'\w+', tok):
            raise ValueError(f"Unexpected token: {tok}")
        return tok

    e = iff()
    if peek() is not None:
        raise ValueError(f"Unexpected token: {peek()}")
    return e

def read_kb():
    s = input("Enter KB sentences (comma-separated, e.g. P=>Q, Q=>R, P): ")
    sents = [parse(x) for x in s.split(',') if x.strip()]
    return sents[0] if len(sents) == 1 else ('and', *sents)

def read_alpha():
    return parse(input("Enter query alpha (e.g. R): "))

def pl_true(sentence, model):
    if isinstance(sentence, str):
        return model.get(sentence, False)
    op = sentence[0]
    if op == 'not':
        return not pl_true(sentence[1], model)
    elif op == 'and':
        return all(pl_true(arg, model) for arg in sentence[1:])
    elif op == 'or':
        return any(pl_true(arg, model) for arg in sentence[1:])
    elif op == 'implies':
        return (not pl_true(sentence[1], model)) or pl_true(sentence[2], model)
    elif op == 'iff':
        return pl_true(sentence[1], model) == pl_true(sentence[2], model)
    return False

def get_symbols(sentence):
    if isinstance(sentence, str):
        return {sentence}
    symbols = set()
    for arg in sentence[1:]:
        symbols.update(get_symbols(arg))
    return symbols

def tt_entails(kb, alpha):
    symbols = sorted(get_symbols(kb) | get_symbols(alpha))
    print(f"{' | '.join(symbols)} || KB    | alpha | Match")
    print("-" * (len(symbols) * 4 + 25))
    return tt_check_all(kb, alpha, symbols, {})

def tt_check_all(kb, alpha, symbols, model):
    if not symbols:
        kb_val = pl_true(kb, model)
        alpha_val = pl_true(alpha, model)
        match = "OK" if (not kb_val or alpha_val) else "FAIL"
        row_vals = [str(model[sym])[0] for sym in sorted(model.keys())]
        print(f"{' | '.join(row_vals)} || {str(kb_val)[0]:<5} | {str(alpha_val)[0]:<5} | {match}")
        return alpha_val if kb_val else True

    P, rest = symbols[0], symbols[1:]

    model_true = model.copy()
    model_true[P] = True
    res_true = tt_check_all(kb, alpha, rest, model_true)

    model_false = model.copy()
    model_false[P] = False
    res_false = tt_check_all(kb, alpha, rest, model_false)

    return res_true and res_false

if __name__ == "__main__":
    kb_expr = read_kb()
    alpha_expr = read_alpha()
    entails = tt_entails(kb_expr, alpha_expr)
    print(f"\nEntailment holds: {entails}")

Enter KB sentences (comma-separated, e.g. P=>Q, Q=>R, P): P=>Q, Q=>R, P 
Enter query alpha (e.g. R): R
P | Q | R || KB    | alpha | Match
-------------------------------------
T | T | T || T     | T     | OK
T | T | F || F     | F     | OK
T | F | T || F     | T     | OK
T | F | F || F     | F     | OK
F | T | T || F     | T     | OK
F | T | F || F     | F     | OK
F | F | T || F     | T     | OK
F | F | F || F     | F     | OK

Entailment holds: True
